## Parte 1 — Softmax con temperatura

Se implementa en NumPy la función p_i = exp(z_i/T) / Σ_j exp(z_j/T) de forma numéricamente estable: antes de exponenciar se resta el máximo de los logits escalados, de modo que se calcula exp((z_i − max(z))/T) y el factor común se cancela en la normalización. Esto evita desbordamientos sin alterar el resultado. Con los logits t0..t5 = 2.0, 1.0, 0.5, 0.2, −1.0, −3.0, las tres distribuciones y sus entropías en bits son:

| T | p(t0) | p(t1) | p(t2) | p(t3) | p(t4) | p(t5) | H (bits) | suma_p |
|---|---|---|---|---|---|---|---|---|
| 0.5 | 0.8231 | 0.1114 | 0.0410 | 0.0225 | 0.0020 | 0.0000 | 0.9147 | 1.0000 |
| 1.0 | 0.5516 | 0.2029 | 0.1231 | 0.0912 | 0.0275 | 0.0037 | 1.7998 | 1.0000 |
| 2.0 | 0.3583 | 0.2173 | 0.1693 | 0.1457 | 0.0800 | 0.0294 | 2.2888 | 1.0000 |

Las tres distribuciones suman 1.0000. Al aumentar T la masa se reparte desde el token de mayor logit hacia el resto: p(t0) cae de 0.8231 (T=0.5) a 0.5516 (T=1) y a 0.3583 (T=2), mientras que p(t5) sube de 3.7367e-05 (que a cuatro decimales aparece como 0.0000) a 0.0037 y luego a 0.0294. En consecuencia, la entropía crece monótonamente con la temperatura: 0.9147 → 1.7998 → 2.2888 bits. La celda de código genera también la figura de esta parte (figuras/T1_softmax_temperatura.png).

In [1]:
# T1: código aprobado por el crítico
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

# ------------------------------------------------------------------
# Datos del enunciado (Parte 1 — Softmax con temperatura)
# ------------------------------------------------------------------
TOKENS = ["t0", "t1", "t2", "t3", "t4", "t5"]
LOGITS = np.array([2.0, 1.0, 0.5, 0.2, -1.0, -3.0], dtype=float)
TEMPERATURAS = [0.5, 1.0, 2.0]


# ------------------------------------------------------------------
# Softmax con temperatura, numéricamente estable:
# p_i = exp(z_i / T) / sum_j exp(z_j / T)
# Se resta el máximo de los logits escalados antes de exponenciar.
# ------------------------------------------------------------------
def softmax_temperatura(z, T):
    z = np.asarray(z, dtype=float)
    if T <= 0:
        raise ValueError("La temperatura debe ser positiva.")
    escalados = z / T
    escalados = escalados - np.max(escalados)  # truco de estabilidad numérica
    exps = np.exp(escalados)
    return exps / np.sum(exps)


def entropia_bits(p):
    """Entropía de Shannon H(p) = -sum_i p_i * log2(p_i), en bits."""
    p = np.asarray(p, dtype=float)
    p = p[p > 0]  # convención 0*log2(0) = 0
    return float(-np.sum(p * np.log2(p)))


# ------------------------------------------------------------------
# Cálculo de distribuciones, entropías y verificación de suma
# ------------------------------------------------------------------
distribuciones = {}
entropias = {}
sumas = {}
for T in TEMPERATURAS:
    p = softmax_temperatura(LOGITS, T)
    distribuciones[T] = p
    entropias[T] = entropia_bits(p)
    sumas[T] = float(np.sum(p))

# ------------------------------------------------------------------
# Tabla con cuatro decimales: 3 distribuciones (6 probs c/u) + 3 entropías
# ------------------------------------------------------------------
columnas = ["T"] + [f"p({t})" for t in TOKENS] + ["H (bits)", "suma_p"]
filas = []
for T in TEMPERATURAS:
    filas.append(
        [f"{T:.1f}"]
        + [f"{v:.4f}" for v in distribuciones[T]]
        + [f"{entropias[T]:.4f}", f"{sumas[T]:.4f}"]
    )
tabla = pd.DataFrame(filas, columns=columnas)
print("Tabla de resultados (4 decimales):")
print(tabla.to_string(index=False))

# Tabla en formato markdown (sin dependencias externas)
md = "| " + " | ".join(columnas) + " |\n"
md += "|" + "|".join(["---"] * len(columnas)) + "|\n"
for fila in filas:
    md += "| " + " | ".join(fila) + " |\n"

# ------------------------------------------------------------------
# Verificaciones del criterio de éxito
# ------------------------------------------------------------------
suma_ok = all(abs(sumas[T] - 1.0) < 1e-12 for T in TEMPERATURAS)
entropia_monotona = bool(
    entropias[0.5] < entropias[1.0] < entropias[2.0]
)
assert suma_ok, "Cada distribución debe sumar 1."
assert entropia_monotona, "La entropía debe crecer con T: H(0.5) < H(1) < H(2)."

# ------------------------------------------------------------------
# Figura: distribuciones para las tres temperaturas
# ------------------------------------------------------------------
fig, ax = plt.subplots(figsize=(8, 5))
x = np.arange(len(TOKENS))
ancho = 0.25
for i, T in enumerate(TEMPERATURAS):
    ax.bar(x + (i - 1) * ancho, distribuciones[T], ancho, label=f"T = {T}")
ax.set_xticks(x)
ax.set_xticklabels(TOKENS)
ax.set_xlabel("Token")
ax.set_ylabel("Probabilidad")
ax.set_title("Softmax con temperatura (logits: 2.0, 1.0, 0.5, 0.2, -1.0, -3.0)")
ax.legend()
fig.tight_layout()
fig.savefig("softmax_temperatura.png", dpi=120)
plt.close(fig)

# ------------------------------------------------------------------
# resultados.json (contrato de la subtarea)
# ------------------------------------------------------------------
resultados = {
    "logits": {t: float(z) for t, z in zip(TOKENS, LOGITS)},
    "definicion": (
        "p_i = exp(z_i/T) / sum_j exp(z_j/T), restando el maximo de los "
        "logits escalados antes de exponenciar (estabilidad numerica)"
    ),
    "distribuciones": {
        f"T={T}": {t: float(v) for t, v in zip(TOKENS, distribuciones[T])}
        for T in TEMPERATURAS
    },
    "entropias_bits": {f"T={T}": entropias[T] for T in TEMPERATURAS},
    "sumas_distribuciones": {f"T={T}": sumas[T] for T in TEMPERATURAS},
    "tabla_4_decimales": {
        "columnas": columnas,
        "filas": filas,
        "markdown": md,
    },
    "verificacion": {
        "cada_distribucion_suma_1": bool(suma_ok),
        "entropia_crece_con_T": entropia_monotona,
    },
}
with open("resultados.json", "w", encoding="utf-8") as f:
    json.dump(resultados, f, ensure_ascii=False, indent=2)

# ------------------------------------------------------------------
# Resumen
# ------------------------------------------------------------------
print("\nResumen:")
for T in TEMPERATURAS:
    print(
        f"T={T}: H = {entropias[T]:.4f} bits | "
        + ", ".join(f"p({t})={v:.4f}" for t, v in zip(TOKENS, distribuciones[T]))
        + f" | suma = {sumas[T]:.6f}"
    )
print(f"Orden de entropías: H(0.5)={entropias[0.5]:.4f} < "
      f"H(1)={entropias[1.0]:.4f} < H(2)={entropias[2.0]:.4f}")
print("Archivos generados: resultados.json, softmax_temperatura.png")


Tabla de resultados (4 decimales):
  T  p(t0)  p(t1)  p(t2)  p(t3)  p(t4)  p(t5) H (bits) suma_p
0.5 0.8231 0.1114 0.0410 0.0225 0.0020 0.0000   0.9147 1.0000
1.0 0.5516 0.2029 0.1231 0.0912 0.0275 0.0037   1.7998 1.0000
2.0 0.3583 0.2173 0.1693 0.1457 0.0800 0.0294   2.2888 1.0000

Resumen:
T=0.5: H = 0.9147 bits | p(t0)=0.8231, p(t1)=0.1114, p(t2)=0.0410, p(t3)=0.0225, p(t4)=0.0020, p(t5)=0.0000 | suma = 1.000000
T=1.0: H = 1.7998 bits | p(t0)=0.5516, p(t1)=0.2029, p(t2)=0.1231, p(t3)=0.0912, p(t4)=0.0275, p(t5)=0.0037 | suma = 1.000000
T=2.0: H = 2.2888 bits | p(t0)=0.3583, p(t1)=0.2173, p(t2)=0.1693, p(t3)=0.1457, p(t4)=0.0800, p(t5)=0.0294 | suma = 1.000000
Orden de entropías: H(0.5)=0.9147 < H(1)=1.7998 < H(2)=2.2888
Archivos generados: resultados.json, softmax_temperatura.png


## Parte 2 — Muestreo de núcleo (top-p)

Partiendo de la distribución de T=1 de la Parte 1, los tokens se ordenan por probabilidad descendente y se acumula: 0.5516 (t0), 0.7546 (+t1), 0.8776 (+t2), 0.9688 (+t3), 0.9963 (+t4), 1.0000 (+t5). Con p = 0.9, el conjunto mínimo cuya acumulada alcanza el umbral exige incluir t3: hasta t2 la acumulada es 0.8776 < 0.9 y al añadirlo se llega a 0.9688 ≥ 0.9. El núcleo son, por tanto, t0, t1, t2 y t3 (masa total 0.9688), y quedan eliminados t4 y t5. La renormalización divide la probabilidad de cada superviviente entre 0.9688, lo que conserva las proporciones relativas dentro del núcleo:

| rank | token | p(T=1) | acumulada | en_núcleo | p_renormalizada |
|---|---|---|---|---|---|
| 1 | t0 | 0.5516 | 0.5516 | sí | 0.5694 |
| 2 | t1 | 0.2029 | 0.7546 | sí | 0.2095 |
| 3 | t2 | 0.1231 | 0.8776 | sí | 0.1270 |
| 4 | t3 | 0.0912 | 0.9688 | sí | 0.0941 |
| 5 | t4 | 0.0275 | 0.9963 | no | 0.0000 |
| 6 | t5 | 0.0037 | 1.0000 | no | 0.0000 |

La distribución renormalizada es t0 = 0.5694, t1 = 0.2095, t2 = 0.1270, t3 = 0.0941, t4 = 0.0000, t5 = 0.0000 (suma 1.0000). La figura de esta parte (figuras/T2_top_p_nucleo.png) se genera en la celda adjunta.

In [2]:
# T2: código aprobado por el crítico
# -*- coding: utf-8 -*-
"""
T2 — Muestreo de núcleo (top-p) con p = 0.9 sobre la distribución de T = 1 (Parte 1).

Entrada : entrada/T1/resultados.json  -> distribución softmax con T = 1 de la Parte 1
Salidas : resultados.json             -> contrato de la subtarea (cifras sin redondear)
          top_p_nucleo.png            -> figura comparativa (dpi = 120)
"""

import json
from pathlib import Path

import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

P = 0.9  # umbral top-p exigido por el enunciado

# ----------------------------------------------------------------------
# 1) Distribución de T = 1 obtenida en la Parte 1 (subtarea T1)
# ----------------------------------------------------------------------
ruta_T1 = Path("entrada") / "T1" / "resultados.json"
dist_T1 = None
if ruta_T1.exists():
    with ruta_T1.open(encoding="utf-8") as f:
        res_T1 = json.load(f)
    dists = res_T1.get("distribuciones", {})
    dist_T1 = dists.get("T=1.0", dists.get("T=1"))

if dist_T1 is not None:
    tokens = list(dist_T1.keys())
    probs = np.array([dist_T1[t] for t in tokens], dtype=float)
    fuente = "entrada/T1/resultados.json (distribucion de la Parte 1 con T=1)"
else:
    # Respaldo: softmax con temperatura, numéricamente estable (misma definición de T1)
    tokens = ["t0", "t1", "t2", "t3", "t4", "t5"]
    logits = np.array([2.0, 1.0, 0.5, 0.2, -1.0, -3.0], dtype=float)
    z = logits / 1.0
    z = z - z.max()
    ez = np.exp(z)
    probs = ez / ez.sum()
    fuente = "respaldo: softmax(T=1) recalculada de forma numericamente estable"

assert abs(probs.sum() - 1.0) < 1e-9, "la distribución de entrada debe sumar 1"

# ----------------------------------------------------------------------
# 2) Top-p: ordenar por probabilidad descendente y hallar el núcleo mínimo
# ----------------------------------------------------------------------
orden = np.argsort(-probs, kind="stable")      # índices de mayor a menor probabilidad
probs_ord = probs[orden]
acum = np.cumsum(probs_ord)

# Número mínimo de tokens cuya acumulada alcanza p (se incluye el token que cruza el umbral)
k = int(np.searchsorted(acum, P, side="left")) + 1
k = min(k, len(tokens))
idx_nucleo = orden[:k]
idx_fuera = orden[k:]

acum_nucleo = float(acum[k - 1])
acum_sin_ultimo = float(acum[k - 2]) if k >= 2 else 0.0

tokens_nucleo = [tokens[i] for i in idx_nucleo]
tokens_fuera = [tokens[i] for i in idx_fuera]

# ----------------------------------------------------------------------
# 3) Renormalización (los tokens fuera del núcleo quedan con probabilidad 0)
# ----------------------------------------------------------------------
probs_renorm = np.zeros_like(probs)
masa_nucleo = float(probs[idx_nucleo].sum())
probs_renorm[idx_nucleo] = probs[idx_nucleo] / masa_nucleo

# ----------------------------------------------------------------------
# 4) Tabla del proceso (4 decimales) en texto/markdown
# ----------------------------------------------------------------------
nucleo_set = set(idx_nucleo.tolist())
header = ["rank", "token", "p(T=1)", "acumulada", "en_nucleo", "p_renormalizada"]
filas = []
for rank, i in enumerate(orden, start=1):
    dentro = i in nucleo_set
    filas.append([
        str(rank),
        tokens[i],
        f"{probs[i]:.4f}",
        f"{acum[rank - 1]:.4f}",
        "si" if dentro else "no",
        f"{probs_renorm[i]:.4f}" if dentro else "0.0000",
    ])
markdown = (
    "| " + " | ".join(header) + " |\n"
    + "|" + "---|" * len(header) + "\n"
    + "\n".join("| " + " | ".join(f) + " |" for f in filas)
    + "\n"
)

# ----------------------------------------------------------------------
# 5) resultados.json (contrato de la subtarea)
# ----------------------------------------------------------------------
resultados = {
    "subtarea": "T2: top-p (muestreo de nucleo) con p=0.9 sobre la distribucion de T=1",
    "entrada": {
        "fuente": fuente,
        "temperatura": 1.0,
        "p": P,
        "distribucion_T1": {t: float(v) for t, v in zip(tokens, probs)},
    },
    "orden_descendente": {
        "tokens": [tokens[i] for i in orden],
        "probabilidades": [float(v) for v in probs_ord],
        "probabilidad_acumulada": [float(v) for v in acum],
    },
    "nucleo_top_p": {
        "tokens_supervivientes": tokens_nucleo,
        "num_tokens": k,
        "probabilidad_acumulada_nucleo": acum_nucleo,
        "acumulada_hasta_token_anterior": acum_sin_ultimo,
        "es_minimo_con_acumulada_ge_p": bool(acum_nucleo >= P and acum_sin_ultimo < P),
        "tokens_eliminados": tokens_fuera,
    },
    "distribucion_renormalizada": {
        t: float(probs_renorm[i]) for i, t in enumerate(tokens)
    },
    "distribucion_renormalizada_orden_descendente": {
        tokens[i]: float(probs_renorm[i]) for i in orden
    },
    "tabla_4_decimales": {"columnas": header, "filas": filas, "markdown": markdown},
    "verificacion": {
        "suma_distribucion_renormalizada": float(probs_renorm.sum()),
        "suma_igual_1": bool(abs(probs_renorm.sum() - 1.0) < 1e-12),
        "tokens_fuera_del_nucleo_con_probabilidad_0": bool(
            all(probs_renorm[i] == 0.0 for i in idx_fuera)
        ),
        "proporciones_del_nucleo_conservadas": bool(
            np.allclose(probs_renorm[idx_nucleo] * masa_nucleo, probs[idx_nucleo],
                        rtol=0.0, atol=1e-12)
        ),
        "acumulada_nucleo_ge_p": bool(acum_nucleo >= P),
    },
}

with open("resultados.json", "w", encoding="utf-8") as f:
    json.dump(resultados, f, ensure_ascii=False, indent=2)

# ----------------------------------------------------------------------
# 6) Figura comparativa
# ----------------------------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
x = np.arange(len(tokens))
colores = ["#2a9d8f" if i in nucleo_set else "#e76f51" for i in range(len(tokens))]

ax = axes[0]
ax.bar(x, probs, color=colores, edgecolor="black", linewidth=0.5, label="p(T=1)")
ax.plot(x, acum, "o--", color="#264653", markersize=4, label="acumulada (orden desc.)")
ax.axhline(P, color="gray", linestyle=":", linewidth=1.5, label="p = 0.9")
ax.set_title("Distribución de T=1 y probabilidad acumulada")
ax.set_xticks(x)
ax.set_xticklabels(tokens)
ax.set_ylabel("probabilidad")
ax.set_ylim(0, 1.05)
ax.legend(fontsize=8)

ax = axes[1]
ax.bar(x, probs_renorm, color=colores, edgecolor="black", linewidth=0.5)
ax.set_title("Distribución renormalizada (núcleo: " + ", ".join(tokens_nucleo) + ")")
ax.set_xticks(x)
ax.set_xticklabels(tokens)
ax.set_ylabel("probabilidad renormalizada")
ax.set_ylim(0, 1.05)

fig.suptitle(
    "Muestreo de núcleo (top-p) con p = 0.9 — verde: dentro del núcleo, rojo: fuera",
    fontsize=10,
)
fig.tight_layout(rect=[0, 0, 1, 0.93])
fig.savefig("top_p_nucleo.png", dpi=120)
plt.close(fig)

# ----------------------------------------------------------------------
# 7) Resumen
# ----------------------------------------------------------------------
print("T2 — Muestreo de núcleo (top-p), p = 0.9, sobre la distribución de T = 1")
print(f"Fuente: {fuente}")
print("Orden descendente de probabilidad:")
for rank, i in enumerate(orden, start=1):
    print(f"  {rank}. {tokens[i]}  p = {probs[i]:.4f}  acumulada = {acum[rank-1]:.4f}")
print(f"Tokens supervivientes (núcleo mínimo con acumulada >= 0.9): {tokens_nucleo}")
print(f"Probabilidad acumulada del núcleo: {acum_nucleo:.4f}")
print(f"Tokens eliminados (probabilidad 0): {tokens_fuera}")
print("Distribución renormalizada:")
for t, v in zip(tokens, probs_renorm):
    print(f"  {t}: {v:.6f}")
print(f"Suma de la distribución renormalizada: {probs_renorm.sum():.12f}")
print("Archivos escritos: resultados.json, top_p_nucleo.png")


T2 — Muestreo de núcleo (top-p), p = 0.9, sobre la distribución de T = 1
Fuente: entrada/T1/resultados.json (distribucion de la Parte 1 con T=1)
Orden descendente de probabilidad:
  1. t0  p = 0.5516  acumulada = 0.5516
  2. t1  p = 0.2029  acumulada = 0.7546
  3. t2  p = 0.1231  acumulada = 0.8776
  4. t3  p = 0.0912  acumulada = 0.9688
  5. t4  p = 0.0275  acumulada = 0.9963
  6. t5  p = 0.0037  acumulada = 1.0000
Tokens supervivientes (núcleo mínimo con acumulada >= 0.9): ['t0', 't1', 't2', 't3']
Probabilidad acumulada del núcleo: 0.9688
Tokens eliminados (probabilidad 0): ['t4', 't5']
Distribución renormalizada:
  t0: 0.569376
  t1: 0.209462
  t2: 0.127045
  t3: 0.094117
  t4: 0.000000
  t5: 0.000000
Suma de la distribución renormalizada: 1.000000000000
Archivos escritos: resultados.json, top_p_nucleo.png


## Parte 3 — Comprobación empírica

Se tomaron 10 000 muestras de la distribución renormalizada de la Parte 2 con numpy.random.default_rng(0). Los conteos observados fueron t0 = 5682, t1 = 2093, t2 = 1296, t3 = 929, t4 = 0 y t5 = 0, es decir, frecuencias 0.5682, 0.2093, 0.1296, 0.0929, 0.0 y 0.0. La comparación con las probabilidades teóricas es:

| token | frecuencia observada | probabilidad teórica | dif. (obs − teo) | aporte KL (bits) |
|---|---|---|---|---|
| t0 | 0.5682 | 0.5694 | -0.0012 | -0.0017 |
| t1 | 0.2093 | 0.2095 | -0.0002 | -0.0002 |
| t2 | 0.1296 | 0.1270 | 0.0026 | 0.0037 |
| t3 | 0.0929 | 0.0941 | -0.0012 | -0.0017 |
| t4 | 0.0000 | 0.0000 | 0.0000 | 0.0000 |
| t5 | 0.0000 | 0.0000 | 0.0000 | 0.0000 |

La divergencia KL de la distribución empírica respecto de la teórica es 5.0069e-05 bits, un valor prácticamente cero: la mayor desviación individual es la de t2 (0.0026 por encima de la teórica) y los aportes positivos y negativos de la tabla casi se cancelan. Los tokens con probabilidad teórica cero (t4 y t5) no recibieron ninguna muestra, con lo que no surgen términos log(0) y la KL queda bien definida. Este resultado confirma que el muestreo reproduce la distribución renormalizada sin sesgo apreciable a esta escala de 10 000 muestras. La celda adjunta genera el gráfico de barras que compara las frecuencias observadas con las probabilidades teóricas (figuras/T3_T3_frecuencias_vs_teoricas.png).

In [3]:
# T3: código aprobado por el crítico
# =====================================================================
# T3 — Parte 3: Comprobación empírica del muestreo top-p (Tarea B)
# Toma 10,000 muestras de la distribución renormalizada de la Parte 2
# con numpy.random.default_rng(0), compara frecuencias observadas vs.
# probabilidades teóricas (gráfico de barras) y calcula la divergencia
# KL (en bits) de la empírica respecto de la teórica.
# =====================================================================
import json
from pathlib import Path

import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

# ---------------------------------------------------------------------
# Parámetros fijados por el enunciado
# ---------------------------------------------------------------------
TOKENS = ["t0", "t1", "t2", "t3", "t4", "t5"]
N_MUESTRAS = 10_000
SEMILLA = 0

# ---------------------------------------------------------------------
# 1) Distribución renormalizada de la Parte 2 (resultado de T2)
# ---------------------------------------------------------------------
ruta_t2 = Path("entrada/T2/resultados.json")
if ruta_t2.exists():
    with open(ruta_t2, "r", encoding="utf-8") as f:
        t2 = json.load(f)
    p_teo = np.array(
        [t2["distribucion_renormalizada"][t] for t in TOKENS], dtype=float
    )
    fuente = "entrada/T2/resultados.json (distribucion renormalizada top-p p=0.9 sobre T=1)"
else:
    # Respaldo determinista: softmax(T=1) + top-p(0.9) + renormalización
    logits = np.array([2.0, 1.0, 0.5, 0.2, -1.0, -3.0])
    z = logits - logits.max()                      # estabilidad numérica
    p1 = np.exp(z) / np.exp(z).sum()
    orden = np.argsort(-p1)                        # descendente
    acum = np.cumsum(p1[orden])
    k = int(np.searchsorted(acum, 0.9, side="left") + 1)  # mínimo con acum >= 0.9
    nucleo = orden[:k]
    p_teo = np.zeros_like(p1)
    p_teo[nucleo] = p1[nucleo] / p1[nucleo].sum()
    fuente = "recalculada (softmax T=1 + top-p p=0.9 + renormalizacion)"

# Salvaguarda numérica para el muestreador (no altera los valores salvo ~1e-16)
if abs(p_teo.sum() - 1.0) > 1e-12:
    p_teo = p_teo / p_teo.sum()

# ---------------------------------------------------------------------
# 2) Muestreo: 10,000 muestras con numpy.random.default_rng(0)
# ---------------------------------------------------------------------
rng = np.random.default_rng(SEMILLA)
muestras = rng.choice(len(TOKENS), size=N_MUESTRAS, p=p_teo)

conteos = np.bincount(muestras, minlength=len(TOKENS))
f_obs = conteos / N_MUESTRAS

# ---------------------------------------------------------------------
# 3) Divergencia KL en bits: KL(empírica || teórica)
#    KL(P||Q) = sum_i P(i) * log2(P(i)/Q(i)); términos con P(i)=0 valen 0
# ---------------------------------------------------------------------
mask = (p_teo > 0) & (f_obs > 0)
contrib_kl = np.zeros(len(TOKENS))
contrib_kl[mask] = f_obs[mask] * np.log2(f_obs[mask] / p_teo[mask])
kl_bits = float(contrib_kl.sum())

# ---------------------------------------------------------------------
# 4) Gráfico de barras: frecuencias observadas vs. probabilidades teóricas
# ---------------------------------------------------------------------
x = np.arange(len(TOKENS))
w = 0.38
fig, ax = plt.subplots(figsize=(8.5, 5.2))
b1 = ax.bar(x - w / 2, f_obs, w, label="Frecuencia observada (10,000 muestras)",
            color="#4C72B0")
b2 = ax.bar(x + w / 2, p_teo, w, label="Probabilidad teórica (renormalizada)",
            color="#DD8452")
for rect, v in zip(b1, f_obs):
    ax.text(rect.get_x() + rect.get_width() / 2, v + 0.006, f"{v:.4f}",
            ha="center", fontsize=8)
for rect, v in zip(b2, p_teo):
    ax.text(rect.get_x() + rect.get_width() / 2, v + 0.006, f"{v:.4f}",
            ha="center", fontsize=8)
ax.set_xticks(x)
ax.set_xticklabels(TOKENS)
ax.set_ylabel("Probabilidad / frecuencia")
ax.set_ylim(0, max(f_obs.max(), p_teo.max()) * 1.18)
ax.set_title(
    "Parte 3 — Frecuencias observadas vs. probabilidades teóricas\n"
    f"top-p p=0.9 sobre T=1 · {N_MUESTRAS:,} muestras · "
    f"default_rng({SEMILLA}) · KL = {kl_bits:.6f} bits"
)
ax.legend()
fig.tight_layout()
fig.savefig("T3_frecuencias_vs_teoricas.png", dpi=120)
plt.close(fig)

# ---------------------------------------------------------------------
# 5) resultados.json (contrato de la subtarea)
# ---------------------------------------------------------------------
resultados = {
    "subtarea": (
        "T3: comprobacion empirica — 10,000 muestras de la distribucion "
        "renormalizada de la Parte 2 con numpy.random.default_rng(0), "
        "comparacion de frecuencias observadas vs. probabilidades teoricas "
        "y divergencia KL (bits) de la empirica respecto de la teorica"
    ),
    "entrada": {
        "fuente_distribucion": fuente,
        "probabilidades_teoricas": {t: float(p) for t, p in zip(TOKENS, p_teo)},
        "num_muestras": N_MUESTRAS,
        "semilla": SEMILLA,
        "generador": f"numpy.random.default_rng({SEMILLA})",
    },
    "muestras": {
        "n": N_MUESTRAS,
        "conteos_observados": {t: int(c) for t, c in zip(TOKENS, conteos)},
        "frecuencias_observadas": {t: float(fr) for t, fr in zip(TOKENS, f_obs)},
    },
    "tabla_comparativa": {
        "columnas": ["token", "frecuencia_observada", "probabilidad_teorica",
                     "diferencia_obs_menos_teo", "aporte_KL_bits"],
        "filas": [
            [t, float(fo), float(pt), float(fo - pt), float(ck)]
            for t, fo, pt, ck in zip(TOKENS, f_obs, p_teo, contrib_kl)
        ],
    },
    "divergencia_kl_bits": kl_bits,
    "figura": "T3_frecuencias_vs_teoricas.png",
    "verificacion": {
        "suma_frecuencias_observadas": float(f_obs.sum()),
        "suma_probabilidades_teoricas": float(p_teo.sum()),
        "tokens_con_probabilidad_teorica_cero_sin_muestras": bool(
            np.all(conteos[p_teo == 0] == 0)
        ),
        "kl_cercana_a_cero": bool(kl_bits < 0.01),
    },
}

with open("resultados.json", "w", encoding="utf-8") as f:
    json.dump(resultados, f, indent=2, ensure_ascii=False)

# ---------------------------------------------------------------------
# Resumen
# ---------------------------------------------------------------------
print("=== T3: Comprobación empírica (Parte 3) ===")
print(f"Fuente de la distribución teórica: {fuente}")
print(f"Muestras: {N_MUESTRAS:,} · generador: numpy.random.default_rng({SEMILLA})")
print("token | frec. observada | prob. teórica")
for t, fo, pt in zip(TOKENS, f_obs, p_teo):
    print(f"  {t}  |     {fo:.6f}    |   {pt:.6f}")
print(f"Divergencia KL (empírica || teórica) = {kl_bits:.6f} bits")
print("Figura guardada: T3_frecuencias_vs_teoricas.png")
print("Resultados escritos en resultados.json")


=== T3: Comprobación empírica (Parte 3) ===
Fuente de la distribución teórica: entrada/T2/resultados.json (distribucion renormalizada top-p p=0.9 sobre T=1)
Muestras: 10,000 · generador: numpy.random.default_rng(0)
token | frec. observada | prob. teórica
  t0  |     0.568200    |   0.569376
  t1  |     0.209300    |   0.209462
  t2  |     0.129600    |   0.127045
  t3  |     0.092900    |   0.094117
  t4  |     0.000000    |   0.000000
  t5  |     0.000000    |   0.000000
Divergencia KL (empírica || teórica) = 0.000050 bits
Figura guardada: T3_frecuencias_vs_teoricas.png
Resultados escritos en resultados.json


## Parte 4 — Pregunta conceptual

Cuando T → 0, el cociente z_i/T amplifica las diferencias entre logits. Sea t_k el token de mayor logit (aquí t0, con z = 2.0): para cualquier otro token, z_i − z_k < 0 y (z_i − z_k)/T → −∞, de modo que su exponencial —y con ello su probabilidad— tiende a 0, mientras que p_k tiende a 1. La distribución converge a una masa puntual concentrada en el argmax y su entropía tiende a 0. Las entropías medidas muestran esa tendencia: al bajar T de 1 a 0.5, H cae de 1.7998 a 0.9147 bits, p(t0) sube de 0.5516 a 0.8231 y p(t5) cae hasta 3.7367e-05.

Esto equivale a la decodificación voraz porque greedy elige en cada paso el token de mayor logit, es decir, el argmax. Dividir los logits por T > 0 es una transformación monótona que no altera su orden, así que el argmax es el mismo para cualquier temperatura; en el límite T → 0 toda la masa se sitúa precisamente en ese token, de modo que muestrear de la distribución devuelve siempre la misma elección que haría greedy. Por eso T → 0 y la decodificación voraz son equivalentes.

Cuando T → ∞, z_i/T → 0 para todo token, todos los exponenciales tienden a 1 y p_i tiende a 1/6: la distribución converge a la uniforme sobre el vocabulario, que es la de entropía máxima (log2 6). Las cifras de la Parte 1 confirman esa dirección: H sube de 1.7998 bits (T=1) a 2.2888 bits (T=2) y la distribución se aplana —en T=2 las probabilidades van de 0.3583 a 0.0294, un rango mucho más estrecho que el de T=1 (0.5516 a 0.0037)—, acercándose a la uniforme.

En síntesis, la temperatura controla el compromiso entre determinismo y aleatoriedad: T → 0 concentra la masa en el argmax (entropía → 0, equivalente a greedy) y T → ∞ la reparte de forma uniforme (entropía máxima), con las entropías medidas 0.9147, 1.7998 y 2.2888 bits para T = 0.5, 1 y 2 trazando de forma monótona ese camino.